### Module 7: Data Wrangling with Pandas

#### CPE311 Computational Thinking with Python

Submitted by: Bermudez, Lorenzo Gabriel <br>
Performed on: 02/24/2026 <br>
Submitted on: 02/24/2026 <br><br>
Submitted to: Engr. Roman M. Richard

### 7.1 Supplementary Activity

Using the datasets provided, perform the following exercises:

#### Exercise 1

We want to look at data for the Facebook, Apple, Amazon, Netflix, and Google (FAANG) stocks, but we were given each asa  separative CSV file. Combine them into a single file and store the dataframe of the FAANG data as faang for the rest of the exercises. <br>
1. Read each file in. <br>
2. Add a column to each dataframe, called ticker symbol it is for (Apple's is AAPL, for example). This is how you look up a stock. Each file's name is also the ticker symbol, so be suer to capitalize it. <br>
3. Append them together into a single dataframe. <br>
4. Save the result in a CSV file called faang.csv 

In [4]:
import pandas as pd

tickers = ['fb', 'aapl', 'amzn', 'nflx', 'goog']

faang_list = []

for ticker in tickers:
    df = pd.read_csv(f"C:\\Users\\Enzo\\Computational Thinking with Python\\CPE-311-Computational-Thinking-with-Python\\Datasets HOA 7.1\\{ticker}.csv")
    
    df['ticker'] = ticker.lower()
    
    faang_list.append(df)

# 3. Append together to a single dataframe
faang = pd.concat(faang_list, sort=False)

faang.to_csv('faang.csv', index=False)

print(f"Combined shape: {faang.shape}")
print(faang.sample(5)) # Shows 5 random rows to verify ticker symbols

Combined shape: (1255, 7)
           date       open       high        low      close    volume ticker
231  2018-11-30   179.5200   179.5598   176.2739   177.8173  39531549   aapl
67   2018-04-10  1431.9900  1438.3800  1415.7000  1436.2200   4280144   amzn
131  2018-07-11  1737.9900  1756.9600  1734.0000  1755.0000   3209782   amzn
123  2018-06-28   182.0379   184.1242   181.7412   183.4222  17365235   aapl
226  2018-11-23  1030.0000  1037.5900  1022.4000  1023.8800    691462   goog


#### Exercise 2

- With faang, use type conversion to change the data column into a datetime and the volume column into integers. Then, sort by date and ticker. <br>
- Find the seven rows with the highest value for volume. <br>
- Right now, the data is somewhere between long and wide format. Use melt() to make it completely long format. Hint: date and ticker are our ID variables (they uniqely identify each row). We need to melt the rest so that we don't have separate columns for open, high, low, close, and volume.

In [5]:
# Converts date column as an object
faang['date'] = pd.to_datetime(faang['date'])

# Convert 'volume' to integers 
faang['volume'] = faang['volume'].astype(int)

# Sort the data by date first
faang = faang.sort_values(by=['date', 'ticker'])

top_seven_volume = faang.nlargest(7, 'volume')
print("Top 7 rows with highest volume:")
print(top_seven_volume)

# open, high, low, close, and volume melting 
faang_long = faang.melt(
    id_vars=['date', 'ticker'], 
    value_vars=['open', 'high', 'low', 'close', 'volume'],
    var_name='measurement', 
    value_name='value'
)

print("\nFirst few rows of the 'Long' format data:")
print(faang_long.head())

Top 7 rows with highest volume:
          date      open      high       low     close     volume ticker
142 2018-07-26  174.8900  180.1300  173.7500  176.2600  169803668     fb
53  2018-03-20  167.4700  170.2000  161.9500  168.1500  129851768     fb
57  2018-03-26  160.8200  161.1000  149.0200  160.0600  126116634     fb
54  2018-03-21  164.8000  173.4000  163.3000  169.3900  106598834     fb
182 2018-09-21  219.0727  219.6482  215.6097  215.9768   96246748   aapl
245 2018-12-21  156.1901  157.4845  148.9909  150.0862   95744384   aapl
212 2018-11-02  207.9295  211.9978  203.8414  205.8755   91328654   aapl

First few rows of the 'Long' format data:
        date ticker measurement      value
0 2018-01-02   aapl        open   166.9271
1 2018-01-02   amzn        open  1172.0000
2 2018-01-02     fb        open   177.6800
3 2018-01-02   goog        open  1048.3400
4 2018-01-02   nflx        open   196.1000


#### Exercise 3

- Using web scraping, search the list of the hospitals, their address and contact information. Save the list in a new csv file, hospitals.csv. <br>
- Using the generated hospitals.csv, convert the csv file into pandas dataframe. Prepare the data using the data necessary preprocessing techniques. 

In [14]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_hospitals_in_the_Philippines"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Edg/120.0.0.0'}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    html_data = StringIO(response.text)
    all_tables = pd.read_html(html_data)

    target_df = None
    for df in all_tables:
        if 'Name of Hospital' in df.columns:
            target_df = df
            break

    if target_df is not None:
        # Extract existing columns
        hospitals_df = target_df[['Name of Hospital', 'Location', 'Class']].copy()
        
        # Preprocessing: Clean citation brackets and whitespace
        for col in hospitals_df.columns:
            hospitals_df[col] = hospitals_df[col].astype(str).str.replace(r'\[.*\]', '', regex=True).str.strip()

        # Add additional columns to hit 6 columns total
        hospitals_df['Ownership'] = 'Government'
        hospitals_df['Region'] = 'NCR'
        
        # Create a 'City' column by extracting the last part of the Location string
        hospitals_df['City'] = hospitals_df['Location'].str.split(',').str[-1].str.strip()

        # Rename for the final CSV requirements
        hospitals_df.columns = ['hospital_name', 'full_address', 'classification', 'ownership', 'region', 'city']

        # Save to CSV
        hospitals_df.to_csv('hospitals.csv', index=False)

        # Exercise 3 Part 2: Load and display a larger sample
        hospitals_final = pd.read_csv('hospitals.csv')
        
        print(f"Success: 'hospitals.csv' created with {len(hospitals_final.columns)} columns.")
        print("-" * 30)
        # Displaying the first 15 rows to show "more printed data"
        print(hospitals_final.head(15))
    else:
        print("Error: Table 'Name of Hospital' not found.")

except Exception as e:
    print(f"Error: {e}")

Success: 'hospitals.csv' created with 6 columns.
------------------------------
                                   hospital_name  \
0                   Caloocan City Medical Center   
1                             Ospital ng Malabon   
2              San Lorenzo Ruiz General Hospital   
3   Gat Andres Bonifacio Memorial Medical Center   
4                               Ospital ng Tondo   
5      Justice Jose Abad Santos General Hospital   
6                            Ospital ng Sampaloc   
7                          Navotas City Hospital   
8                           Ospital ng Parañaque   
9               Ospital ng Parañaque District II   
10                  Novaliches District Hospital   
11                       San Juan Medical Center   
12                         Army General Hospital   
13                         Manila Naval Hospital   
14                  Taguig City General Hospital   

                                         full_address classification  \
0              

#### 7.2 Conclusion:

In this exercise, I learned how to move from using saved files to pulling live data directly from the internet. I observed that web scraping is a powerful way to get real-world information, but it can be messy. I had a hard time at first with technical "OSErrors" and figuring out how to tell Python exactly which table to look for among so much code. Once I used the specific "Name of Hospital" header to find my data, the process made more sense. I successfully learned how to clean up text, add my own columns, and save everything into a clean CSV file for my project.